# ClinVar Validation

Finds ClinVar variants with **conflicting** submitter classifications, and variants with a **clear, corroborated** classification (Pathogenic or Benign), across the Cardiac_G2P gene panel -- then drives `mainn.ipynb` itself (via `%run -i`, in batch mode) for each one and compares the pipeline's call against ClinVar's.

This deliberately does **not** reimplement `mainn.ipynb`'s classification logic -- it runs the exact same notebook a real interactive session would use, just with `_VALIDATION_ANSWERS` pre-set so every prompt (disease dropdown, variant input, de novo/zygosity, curator-only evidence) answers itself instead of waiting for input. See the "Batch mode" comments in `mainn.ipynb`'s interactive cells for how that works.

This is a small (10-variant) worked example. To scale it up towards the full ~100-variant validation task, use `find_conflicting_variants`/`find_clear_variants` below to pull fresh candidate pools and extend `VALIDATION_SET`.


In [ ]:
import requests
import re


In [ ]:
# --------------------------------------------------
# Load g2p_clean / disease_ref / classify_variant etc. directly
# --------------------------------------------------
# Everything below needs classifierr.ipynb's globals (g2p_clean, disease_ref,
# classify_variant, ...). Running the worked 10-variant demo further down
# loads these as a side effect of its first run_variant_through_main1() call
# (which %runs mainn.ipynb, which in turn %runs annotationn.ipynb/
# classifierr.ipynb) -- but if you skip straight to the ~100-variant batch
# without running that demo first, nothing else would ever load them. This
# mirrors mainn.ipynb's own guard so this notebook works standalone either way.
if "g2p_clean" not in globals():
    %run annotationn.ipynb
    %run classifierr.ipynb


## ClinVar search helpers (NCBI eutils)

`find_clear_variants` deliberately does **not** trust ClinVar's `[Clinical_significance]` esearch field to filter by exact significance -- confirmed empirically that combining it with a review-status term returns identical results regardless of whether the term is "pathogenic" or "benign" (esearch's text index for this field doesn't restrict the way you'd expect). Instead it pulls a broad pool by review status alone, then filters to an *exact* "Pathogenic"/"Benign" description client-side against esummary's real data, which is the only reliable source of truth here.

The corroboration bar (`_CORROBORATED_REVIEW_STATUSES`) matches `_is_corroborated` in `classifierr.ipynb`'s PM5 rule exactly, so the "clear" set here is held to the same evidentiary standard the pipeline itself already uses.

In [ ]:
import time

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

_CORROBORATED_REVIEW_STATUSES = {
    "criteria provided, multiple submitters, no conflicts",
    "reviewed by expert panel",
    "practice guideline",
}


def _gene_term(genes):
    return "(" + " OR ".join(f"{g}[gene]" for g in genes) + ")"


_NCBI_TRANSIENT_STATUS = {429, 500, 502, 503, 504}


def _ncbi_get(url, params, max_retries=6, backoff_base=2.0):
    """
    Retries on 429 (rate limit) and 5xx with exponential backoff (2s, 4s,
    8s...). Unauthenticated eutils access is capped at 3 req/sec, but even a
    correctly-paced sequence of many batched calls (e.g. pulling a
    ~45,000-record pool for find_graded_variants) can still trip a 429 partway
    through -- confirmed live: a 45-id esummary batch failed with 429 after
    ~1000 prior batches at the existing 0.34s pace. A 429/5xx here means
    "wait and retry", not "the request is malformed" -- a genuine 4xx
    (bad query syntax) is NOT retried and raises immediately.
    """
    for attempt in range(1, max_retries + 1):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code in _NCBI_TRANSIENT_STATUS and attempt < max_retries:
            wait = backoff_base ** attempt
            print(f"  NCBI HTTP {r.status_code}; retrying in {wait:.0f}s ({attempt}/{max_retries})")
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r
    r.raise_for_status()
    return r


def _esearch(term, retmax):
    r = _ncbi_get(f"{EUTILS}/esearch.fcgi",
                  params={"db": "clinvar", "term": term, "retmax": retmax, "retmode": "json"})
    return r.json()["esearchresult"]["idlist"]


def _esummary(ids, batch_size=45, pause=0.34):
    """Batched, paced esummary lookup -- stays comfortably under NCBI's unauthenticated 3 req/sec limit."""
    out = {}
    for i in range(0, len(ids), batch_size):
        batch = ids[i:i + batch_size]
        r = _ncbi_get(f"{EUTILS}/esummary.fcgi",
                      params={"db": "clinvar", "id": ",".join(batch), "retmode": "json"})
        res = r.json()["result"]
        out.update({uid: doc for uid, doc in res.items() if uid != "uids"})
        time.sleep(pause)
    return out


def _extract_cdna_hgvs(title, gene):
    """'NM_000256.3(MYBPC3):c.405A>G (p.Lys135=)' -> 'NM_000256.3(MYBPC3):c.405A>G'"""
    m = re.match(r"^(NM_\d+\.\d+)\([A-Z0-9]+\):(c\.[^\s]+)", title or "")
    return f"{m.group(1)}({gene}):{m.group(2)}" if m else None


def find_conflicting_variants(genes, retmax=60):
    """ClinVar variants explicitly classified 'Conflicting classifications of pathogenicity'."""
    term = f'{_gene_term(genes)} AND "conflicting classifications of pathogenicity"[Clinical_significance]'
    summaries = _esummary(_esearch(term, retmax))

    hits = []
    for uid, doc in summaries.items():
        gc = doc.get("germline_classification", {}) or {}
        if "conflicting" not in str(gc.get("description") or "").lower():
            continue
        genes_field = doc.get("genes") or []
        gene = genes_field[0].get("symbol") if genes_field else None
        hgvs = _extract_cdna_hgvs(doc.get("title"), gene) if gene else None
        if gene and hgvs:
            hits.append({"uid": uid, "gene": gene, "hgvs": hgvs,
                         "clinvar_significance": gc.get("description"),
                         "clinvar_review_status": gc.get("review_status")})
    return hits


def find_clear_variants(genes, retmax=600):
    """ClinVar variants with a single, corroborated Pathogenic or Benign call. See note above on why this filters client-side."""
    review_term = (
        '("criteria provided, multiple submitters, no conflicts"[Review_status] '
        'OR "reviewed by expert panel"[Review_status] '
        'OR "practice guideline"[Review_status])'
    )
    summaries = _esummary(_esearch(f"{_gene_term(genes)} AND {review_term}", retmax))

    pathogenic, benign = [], []
    for uid, doc in summaries.items():
        gc = doc.get("germline_classification", {}) or {}
        desc = str(gc.get("description") or "").strip().lower()
        review = str(gc.get("review_status") or "").strip()
        if review not in _CORROBORATED_REVIEW_STATUSES or desc not in ("pathogenic", "benign"):
            continue
        genes_field = doc.get("genes") or []
        gene = genes_field[0].get("symbol") if genes_field else None
        hgvs = _extract_cdna_hgvs(doc.get("title"), gene) if gene else None
        if not (gene and hgvs):
            continue
        rec = {"uid": uid, "gene": gene, "hgvs": hgvs,
               "clinvar_significance": gc.get("description"), "clinvar_review_status": review}
        (pathogenic if desc == "pathogenic" else benign).append(rec)
    return {"pathogenic": pathogenic, "benign": benign}


_GRADED_TIER_KEYS = {
    "pathogenic":             "pathogenic",
    "likely pathogenic":      "likely_pathogenic",
    "uncertain significance": "uncertain_significance",
    "likely benign":          "likely_benign",
    "benign":                 "benign",
}


def find_graded_variants(genes, retmax=45000):
    """
    Like find_clear_variants, but keeps all 5 ACMG/AMP-style ClinVar tiers
    (Pathogenic, Likely Pathogenic, Uncertain significance, Likely Benign,
    Benign) instead of only the two extremes -- needed to plot pipeline
    concordance across the FULL classification scale, not just a P-vs-B
    check. Held to the same corroborated-review-status bar as
    find_clear_variants (_CORROBORATED_REVIEW_STATUSES), so a VUS call
    here is exactly as well-supported, in ClinVar's own terms, as a
    Pathogenic or Benign one -- this only broadens which DESCRIPTIONS are
    kept, not how rigorously each one has to be reviewed.

    Combined categories ("pathogenic/likely pathogenic",
    "benign/likely benign") are excluded, same as find_clear_variants --
    they don't cleanly map to a single tier and keeping the bar strict is
    more important here than maximizing pool size.
    """
    review_term = (
        '("criteria provided, multiple submitters, no conflicts"[Review_status] '
        'OR "reviewed by expert panel"[Review_status] '
        'OR "practice guideline"[Review_status])'
    )
    summaries = _esummary(_esearch(f"{_gene_term(genes)} AND {review_term}", retmax))

    pool = {tier: [] for tier in _GRADED_TIER_KEYS.values()}
    for uid, doc in summaries.items():
        gc = doc.get("germline_classification", {}) or {}
        desc = str(gc.get("description") or "").strip().lower()
        review = str(gc.get("review_status") or "").strip()
        if review not in _CORROBORATED_REVIEW_STATUSES or desc not in _GRADED_TIER_KEYS:
            continue
        genes_field = doc.get("genes") or []
        gene = genes_field[0].get("symbol") if genes_field else None
        hgvs = _extract_cdna_hgvs(doc.get("title"), gene) if gene else None
        if not (gene and hgvs):
            continue
        rec = {"uid": uid, "gene": gene, "hgvs": hgvs,
               "clinvar_significance": gc.get("description"), "clinvar_review_status": review}
        pool[_GRADED_TIER_KEYS[desc]].append(rec)
    return pool


## Pipeline runner

Drives `mainn.ipynb` directly via `%run -i` -- the `-i` flag runs it *in this notebook's own namespace*, so `context`/`result` (defined inside `mainn.ipynb`) are readable here immediately afterward, and `mainn.ipynb`'s own `if "g2p_clean" not in globals()` guard means the (slow) `annotationn.ipynb`/`classifierr.ipynb` load only happens once, not on every variant.

Manual/clinical fields (de novo, zygosity, phasing, segregation, PP4, BS2, BP5) are all left at neutral/unknown defaults here -- this is a retrospective ClinVar comparison, not a real patient case, so it would be misleading to fabricate case-specific facts the pipeline can't actually know from ClinVar alone. Override any of them via `**extra_answers`.


In [ ]:
def run_variant_through_main1(hgvs, disease_short_code=None, **extra_answers):
    """
    Drives the ACTUAL mainn.ipynb notebook via `%run -i` -- not a separate
    reimplementation of its logic -- so this validation always exercises
    exactly the same code path a real interactive session would use.
    Setting _VALIDATION_ANSWERS beforehand makes every interactive cell in
    mainn.ipynb (disease dropdown, variant input, de novo/zygosity,
    curator-only evidence) take its answer from this dict instead of
    prompting/waiting for widget interaction -- see the "Batch mode"
    comments in those cells.

    disease_short_code is one of disease_ref's codes (e.g. "HCM-FAM",
    "LQTS", "ARVC") -- pick whichever matches the variant's gene-disease
    association in Cardiac_G2P (check g2p_clean if unsure); pass None to
    fall back to "Unknown / other" for genes outside the 8 modeled diseases.

    Every patient-specific field defaults to "unknown" -- not "n"/negative --
    since a retrospective ClinVar variant has no real patient behind it, and
    an explicit "no" is a different (and false) claim from "we don't know."
    "unknown" and "n" happen to fire identically for every one of these
    codes (confirmed against apply_ps2_pm6_rule and _manual_curator_hits in
    classifierr.ipynb -- both only ever fire on an explicit affirmative), so
    this changes what gets truthfully recorded, not what fires. zygosity is
    the one exception: it has no "unknown" state in the pipeline (only
    het/hom, since it feeds a real BA1/BS1 threshold adjustment for
    biallelic genes), so it stays defaulted to the more common "het".

    extra_answers can override any of the manual/curator defaults (e.g.
    de_novo_answer="y", zygosity_answer="hom", phasing_answer="y",
    phasing_relationship="trans", segregation_answer="y",
    segregation_result="segregates", pp4_answer="y", bs2_answer="y",
    bp5_answer="y").
    """
    global _VALIDATION_ANSWERS
    _VALIDATION_ANSWERS = {
        "variant": hgvs,
        "disease_short_code": disease_short_code,
        "de_novo_answer": "unknown",
        "zygosity_answer": "het",
        "phasing_answer": "unknown",
        "segregation_answer": "unknown",
        "pp4_answer": "unknown",
        "bs2_answer": "unknown",
        "bp5_answer": "unknown",
        **extra_answers,
    }

    get_ipython().run_line_magic("run", "-i mainn.ipynb")

    gdp = context["gene_disease_pair"]
    return {
        "hgvs": hgvs,
        "gene": context.get("gene_symbol"),
        "pipeline_classification": result["classification"],
        "vus_subtier": result.get("vus_subtier"),
        "posterior_probability": result["posterior_probability"],
        "tavtigian_score": result["combined_score"],
        "evidence_codes": result["evidence_codes"],
        "gene_disease_pair_confirmed": gdp["pair_found"],
        "gene_disease_pair_reason": gdp.get("reason"),
    }


## Worked example: 5 conflicting + 5 clear (2026-07-21 run)

These 10 were selected from live `find_conflicting_variants`/`find_clear_variants` output. `disease_short_code` was chosen by checking each gene's actual `referral_indication` (and `hcm_subtype` where relevant, e.g. FLNC/GAA are curated Syndromic HCM) in `g2p_clean` -- not guessed.

In [ ]:
import pandas as pd

VALIDATION_SET = [
    # (clinvar_set, hgvs, disease_short_code, clinvar_significance)
    ("conflicting", "NM_000256.3(MYBPC3):c.1305G>A",   "HCM-FAM", "Conflicting classifications of pathogenicity"),
    ("conflicting", "NM_000257.4(MYH7):c.1956G>T",     "HCM-FAM", "Conflicting classifications of pathogenicity"),
    ("conflicting", "NM_001035.3(RYR2):c.11947C>T",    "CPVT",    "Conflicting classifications of pathogenicity"),
    ("conflicting", "NM_004415.4(DSP):c.8526_8537del", "ARVC",    "Conflicting classifications of pathogenicity"),
    ("conflicting", "NM_000238.4(KCNH2):c.1597G>A",    "LQTS",    "Conflicting classifications of pathogenicity"),
    ("clear",       "NM_005070.4(SLC4A3):c.1962C>A",   "SQTS",    "Benign"),
    ("clear",       "NM_001458.5(FLNC):c.7501C>T",     "HCM-SYN", "Pathogenic"),
    ("clear",       "NM_000256.3(MYBPC3):c.2998A>T",   "HCM-FAM", "Pathogenic"),
    ("clear",       "NM_000218.3(KCNQ1):c.1733-1G>A",  "LQTS",    "Pathogenic"),
    ("clear",       "NM_000152.5(GAA):c.1529_1541del", "HCM-SYN", "Pathogenic"),
]

rows = []
for clinvar_set, hgvs, disease_code, cv_sig in VALIDATION_SET:
    r = run_variant_through_main1(hgvs, disease_code)
    r["clinvar_set"] = clinvar_set
    r["clinvar_significance"] = cv_sig
    rows.append(r)

results_df = pd.DataFrame(rows)
results_df[["clinvar_set", "gene", "hgvs", "clinvar_significance",
            "pipeline_classification", "posterior_probability", "tavtigian_score", "evidence_codes"]]


In [ ]:
# Presentation-formatted table (S.No / short variant / Gene / ClinVar /
# CardioClassifier v2 call+posterior / Tavtigian score / Evidence codes) --
# same shape as the slide table, generated straight from results_df instead
# of transcribed by hand.
def _short_variant(hgvs):
    """'NM_005070.4(SLC4A3):c.1962C>A' -> 'c.1962C>A'"""
    return hgvs.split(":")[-1]


presentation_df = pd.DataFrame({
    "S.No":              range(1, len(results_df) + 1),
    "Variant":           results_df["hgvs"].map(_short_variant),
    "Gene":              results_df["gene"],
    "ClinVar":           results_df["clinvar_significance"],
    "Cardioclassifier v2": [
        f"{cls} ({post:.3f})"
        for cls, post in zip(results_df["pipeline_classification"], results_df["posterior_probability"])
    ],
    "Tavtigian score":   results_df["tavtigian_score"],
    "Evidence":          results_df["evidence_codes"].map(", ".join),
})
presentation_df


In [ ]:
# Always derived live from g2p_clean, not hardcoded -- stays in sync with
# whatever Cardiac_G2P_cleaned_HCM_syndromic.csv actually contains.
ALL_PANEL_GENES = sorted(g2p_clean["gene_symbol"].dropna().unique().tolist())
print(f"{len(ALL_PANEL_GENES)} genes in the panel:", ALL_PANEL_GENES)


In [ ]:
# --------------------------------------------------
# Auto-inferring disease_short_code for a retrospective ClinVar variant
# --------------------------------------------------
# Unlike a real patient (who has an actual clinical diagnosis), a ClinVar
# record only gives us a gene -- and 21 of the ~97 panel genes are curated
# in Cardiac_G2P for MORE THAN ONE distinct disease (e.g. KCNQ1: LQTS/SQTS/
# JLNS; SCN5A: BrS/LQTS/DCM), so "the" disease for a gene is often genuinely
# ambiguous without phenotype data ClinVar doesn't record. This picks the
# gene's Cardiac_G2P row with the strongest gene-disease validity tier
# (Definitive > Strong > Moderate > Limited) as the best single guess, and
# reports whether the pick was ambiguous so genuinely uncertain rows can be
# inspected or excluded from a stricter concordance analysis later.

_VALIDITY_RANK = {"definitive": 4, "strong": 3, "moderate": 2, "limited": 1}


def infer_disease_short_code(gene_symbol, g2p_df=None, disease_ref_df=None):
    """Returns (disease_short_code_or_None, is_ambiguous, candidate_codes)."""
    g2p_df = g2p_df if g2p_df is not None else g2p_clean
    disease_ref_df = disease_ref_df if disease_ref_df is not None else disease_ref

    rows = g2p_df[g2p_df["gene_symbol"].astype(str).str.upper() == str(gene_symbol).upper()]
    if rows.empty:
        return None, False, []

    candidates = []
    for _, row in rows.iterrows():
        match = disease_ref_df[disease_ref_df["referral_indication"] == row["referral_indication"]]
        if len(match) > 1:  # HCM split into HCM-FAM/HCM-SYN -- use this row's own subtype to pick
            match = match[match["hcm_subtype"] == row.get("hcm_subtype")]
        if match.empty:
            continue
        rank = _VALIDITY_RANK.get(str(row.get("gene_disease_validity", "")).strip().lower(), 0)
        candidates.append((match.iloc[0]["dis_name"], rank))

    if not candidates:
        return None, False, []

    unique_codes = sorted({code for code, _rank in candidates})
    is_ambiguous = len(unique_codes) > 1
    best_code = sorted(candidates, key=lambda c: (-c[1], c[0]))[0][0]
    return best_code, is_ambiguous, unique_codes


## Scaling up: ~100-variant validation batch across all 5 ClinVar tiers

`infer_disease_short_code` (above) resolves each variant's `disease_short_code` automatically from its ClinVar-reported gene, so this can run unattended across a large pool. `disease_short_code` only affects the disease-scoped mechanism rules (PVS1/BP1/BS2/BP2 -- see `build_classification_context` in `classifierr.ipynb`); population/computational thresholds (PM2, BA1/BS1, PP3/BP4, etc.) are looked up per-gene via ClinGen CSpec and are unaffected by which candidate disease is picked when a gene is ambiguous.

This batch draws from all 5 ACMG/AMP ClinVar tiers (Pathogenic, Likely Pathogenic, Uncertain significance, Likely Benign, Benign), not just the Pathogenic/Benign extremes -- see `find_graded_variants` above -- so the concordance plot shows the pipeline's behaviour across the full classification scale, including the clinically important VUS middle.

The batch runner below is resumable (checkpoints every variant to `outputs/validation_clear_batch.jsonl`) since each variant triggers a full VEP + gnomAD + ClinVar round trip, and the pool itself can be large even when TARGET_TOTAL is small -- re-running the cell after an interruption picks up where it left off instead of starting over.


In [ ]:
# --------------------------------------------------
# Build the variant pool, across all 5 ClinVar tiers
# --------------------------------------------------
# Uses find_graded_variants (not find_clear_variants) so the concordance
# plot gets points across the full ACMG/AMP scale -- Uncertain significance
# and the two "Likely" tiers included, not just the Pathogenic/Benign
# extremes.
#
# Splits TARGET_TOTAL evenly across the 5 tiers, and redistributes any
# shortfall (chiefly from the rare Benign tier) across whichever tiers have
# spare capacity, so the achieved total still lands close to TARGET_TOTAL
# even though the tiers are wildly unbalanced in ClinVar itself.
import json
from pathlib import Path

TIER_ORDER = ["pathogenic", "likely_pathogenic", "uncertain_significance", "likely_benign", "benign"]
TARGET_TOTAL = 100
target_per_tier = TARGET_TOTAL // len(TIER_ORDER)

# Skip the live pool-build entirely if the existing checkpoint already has
# >= target_per_tier successfully-classified variants in every tier -- the
# esummary pool-build (retmax=45000, sized for a 1000-variant target's rare-
# Benign yield -- sampling the first 3000 raw hits gave ~1744 Uncertain
# significance, ~877 Likely Benign, ~157 Likely Pathogenic, ~110 Pathogenic,
# and only ~3 strict Benign) is a several-minute paced-NCBI-call cost that a
# smaller, already-satisfied target shouldn't have to re-pay.
_CLINVAR_TO_TIER_KEY = {
    "Pathogenic": "pathogenic", "Likely pathogenic": "likely_pathogenic",
    "Uncertain significance": "uncertain_significance",
    "Likely benign": "likely_benign", "Benign": "benign",
}
_existing_checkpoint = Path("outputs/validation_clear_batch.jsonl")
_existing_counts = {tier: 0 for tier in TIER_ORDER}
if _existing_checkpoint.exists():
    with _existing_checkpoint.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            if "error" in rec:
                continue
            tier = _CLINVAR_TO_TIER_KEY.get(rec.get("clinvar_significance"))
            if tier:
                _existing_counts[tier] += 1

if all(_existing_counts[t] >= target_per_tier for t in TIER_ORDER):
    print(f"Checkpoint already has >= {target_per_tier} per tier "
          f"({_existing_counts}) -- skipping the live ~45k-record pool fetch.")
    batch_variants = []
else:
    graded_pool = find_graded_variants(ALL_PANEL_GENES, retmax=45000)
    print("Raw pool:", {tier: len(graded_pool[tier]) for tier in TIER_ORDER})

    selected = {tier: graded_pool[tier][:min(len(graded_pool[tier]), target_per_tier)] for tier in TIER_ORDER}

    shortfall = TARGET_TOTAL - sum(len(v) for v in selected.values())
    spare_tiers = [t for t in TIER_ORDER if len(selected[t]) < len(graded_pool[t])]
    while shortfall > 0 and spare_tiers:
        for t in list(spare_tiers):
            if shortfall <= 0:
                break
            if len(selected[t]) < len(graded_pool[t]):
                selected[t].append(graded_pool[t][len(selected[t])])
                shortfall -= 1
            if len(selected[t]) >= len(graded_pool[t]):
                spare_tiers.remove(t)

    batch_variants = [v for tier in TIER_ORDER for v in selected[tier]]
    print("Batch selected:", {tier: len(selected[tier]) for tier in TIER_ORDER},
          f"({len(batch_variants)} total)")
    if len(selected["benign"]) < target_per_tier:
        print("Note: strict ClinVar 'Benign' calls are inherently rare for this gene panel "
              "(most benign-leaning submissions land as 'Likely Benign') -- the shortfall was "
              "redistributed to the other tiers rather than leaving the batch under target.")


In [ ]:
# --------------------------------------------------
# Resumable batch runner
# --------------------------------------------------
# Every completed variant is appended to a JSONL checkpoint file
# immediately (not held in memory until the end) and flushed to disk right
# away. Re-running this cell skips any hgvs already SUCCESSFULLY recorded
# there -- but a previously FAILED variant is retried, not skipped, since
# annotationn.ipynb's VEP call now has its own retry/backoff for transient
# errors (500/502/503/504, timeouts) and a variant that failed before a fix
# like that landed deserves another chance rather than being stuck as a
# permanent failure. A per-variant try/except means one bad variant is
# logged and skipped rather than aborting the whole batch.
import time
from pathlib import Path

CHECKPOINT_PATH = Path("outputs/validation_clear_batch.jsonl")
PAUSE_BETWEEN_VARIANTS = 0.5  # be polite to NCBI/Ensembl/gnomAD over a long run
CHECKPOINT_PATH.parent.mkdir(exist_ok=True)

succeeded_before = set()
if CHECKPOINT_PATH.exists():
    with CHECKPOINT_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            if "error" not in rec:
                succeeded_before.add(rec["hgvs"])
    print(f"Resuming: {len(succeeded_before)} variants already succeeded in {CHECKPOINT_PATH} "
          f"(previously-failed variants will be retried)")

n_ok, n_failed = 0, 0
with CHECKPOINT_PATH.open("a", encoding="utf-8") as ckpt:
    for i, rec in enumerate(batch_variants, start=1):
        hgvs = rec["hgvs"]
        if hgvs in succeeded_before:
            continue

        gene = rec["gene"]
        disease_code, is_ambiguous, candidates = infer_disease_short_code(gene)

        try:
            result_row = run_variant_through_main1(hgvs, disease_code)
        except Exception as exc:
            n_failed += 1
            ckpt.write(json.dumps({
                "hgvs": hgvs, "gene": gene, "error": str(exc),
                "clinvar_significance": rec["clinvar_significance"],
            }) + "\n")
            ckpt.flush()
            print(f"[{i}/{len(batch_variants)}] FAILED {hgvs}: {exc}")
            time.sleep(PAUSE_BETWEEN_VARIANTS)
            continue

        result_row.update({
            "clinvar_significance":          rec["clinvar_significance"],
            "clinvar_review_status":         rec.get("clinvar_review_status"),
            "disease_short_code_used":       disease_code,
            "disease_short_code_ambiguous":  is_ambiguous,
            "disease_short_code_candidates": candidates,
        })
        ckpt.write(json.dumps(result_row, default=str) + "\n")
        ckpt.flush()
        n_ok += 1

        if i % 25 == 0 or i == len(batch_variants):
            print(f"[{i}/{len(batch_variants)}] {n_ok} ok, {n_failed} failed so far")

        time.sleep(PAUSE_BETWEEN_VARIANTS)

print(f"Done. {n_ok} succeeded, {n_failed} failed this run. Full results in {CHECKPOINT_PATH}")


In [ ]:
# --------------------------------------------------
# Load the checkpoint file and save a clean summary for viewing/reuse
# --------------------------------------------------
_records = []
with CHECKPOINT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            _records.append(json.loads(line))

validation_results_df = pd.DataFrame(_records)

if "error" in validation_results_df.columns:
    failed_df = validation_results_df[validation_results_df["error"].notna()]
    validation_results_df = validation_results_df[validation_results_df["error"].isna()].drop(columns=["error"])
else:
    failed_df = validation_results_df.iloc[0:0]

print(f"{len(validation_results_df)} variants classified successfully, {len(failed_df)} failed")

_summary_cols = ["hgvs", "gene", "clinvar_significance", "pipeline_classification",
                  "vus_subtier", "posterior_probability", "tavtigian_score", "evidence_codes",
                  "disease_short_code_used", "disease_short_code_ambiguous"]

validation_results_df[_summary_cols].to_csv("outputs/validation_clear_summary.csv", index=False)
validation_results_df.to_json("outputs/validation_clear_summary.json", orient="records", indent=2)
print("Saved: outputs/validation_clear_summary.csv, outputs/validation_clear_summary.json")
print("(Full per-code evidence detail for any one variant is in its own "
      "outputs/<variant>_classification.json, written automatically by mainn.ipynb.)")

validation_results_df[_summary_cols].head(20)


## Concordance plot: pipeline vs ClinVar

Both classifications are mapped onto the same 5-tier ACMG/AMP scale (Benign=1 ... Pathogenic=5) so they're directly comparable. Each point is a single variant, jittered in both x and y so individual points stay visible even where thousands share the same (ClinVar tier, pipeline tier) pair rather than overlapping into a solid bar. The dashed line is the ideal `y = x` (perfect agreement); the solid line connects the *actual mean* pipeline tier for each ClinVar truth class -- the closer that line sits to the diagonal, the better the pipeline's average calibration against ClinVar.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# pipeline_classification (classify_variant's own output) uses this exact
# casing/wording -- "VUS", "Likely Pathogenic", "Likely Benign".
_TIER = {"Benign": 1, "Likely Benign": 2, "VUS": 3, "Likely Pathogenic": 4, "Pathogenic": 5}

# ClinVar's raw germline_classification.description strings use different
# casing/wording ("Uncertain significance", not "VUS"; "Likely benign"/
# "Likely pathogenic", lowercase second word) -- confirmed against real
# find_graded_variants output. Mapping clinvar_significance through _TIER
# directly silently dropped every non-Pathogenic/non-Benign ClinVar tier
# (NaN -> removed by dropna below) without erroring, since until this
# batch clinvar_significance only ever held "Pathogenic"/"Benign", which
# happen to match _TIER's casing by coincidence.
_CLINVAR_TIER = {
    "Benign": 1, "Likely benign": 2, "Uncertain significance": 3,
    "Likely pathogenic": 4, "Pathogenic": 5,
}

# Pastel palette -- checked (not eyeballed) against OKLab CVD-simulated and
# normal-vision delta-E, lightness band, and chroma floor: this pair clears
# every hard gate (CVD dE 11.3/12.6 protan/deutan, normal-vision dE 20.5,
# vs targets of >=8 and >=15). Both sit below 3:1 contrast on a light
# surface, which is expected for a pastel pair -- the mitigation is a
# legend entry for each series rather than relying on hue alone, which
# both already have below.
COLOR_POINTS = "#6da7ec"  # sequential-blue step 300 -- same step used in benchmark_validation.ipynb's
                           # pastel submitter-agreement chart
COLOR_MEAN   = "#104281"  # sequential-blue step 650 -- darkest step of the same ramp
                           # benchmark_validation.ipynb's conflicting-variant distribution chart uses,
                           # swapped in place of an earlier off-family magenta so both
                           # notebooks read as one blue-only palette, not two different ones
COLOR_IDEAL  = "#b4b3ac"  # neutral chrome, not data-encoded

plot_df = validation_results_df.copy()
plot_df["clinvar_tier"]  = plot_df["clinvar_significance"].map(_CLINVAR_TIER)
plot_df["pipeline_tier"] = plot_df["pipeline_classification"].map(_TIER)
plot_df = plot_df.dropna(subset=["clinvar_tier", "pipeline_tier"])

# Stratified sample: up to PLOT_PER_TIER per ClinVar tier, not the whole
# accumulated checkpoint -- validation_results_df carries every variant
# ever successfully run (1703 Benign / 779 Pathogenic from earlier batches,
# dwarfing the newer 20-per-tier Likely Pathogenic/VUS/Likely Benign), so
# plotting it unfiltered buries the very tiers this run was meant to add.
PLOT_PER_TIER = 20
plot_df = (
    plot_df.groupby("clinvar_tier", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), PLOT_PER_TIER), random_state=0))
)

# Save the actual 100-variant sample this plot is built from -- previously
# this only ever existed as an in-memory plot_df; validation_clear_summary.csv
# is the full accumulated checkpoint (2500+ rows), not this stratified sample,
# so there was no standalone file matching what the plot actually shows.
_sample_cols = ["hgvs", "gene", "clinvar_significance", "pipeline_classification",
                "posterior_probability", "tavtigian_score", "evidence_codes",
                "disease_short_code_used"]
_sample_cols = [c for c in _sample_cols if c in plot_df.columns]
plot_df[_sample_cols].to_csv("outputs/validation_concordance_100_sample.csv", index=False)
print(f"Saved {len(plot_df)}-variant plot sample: outputs/validation_concordance_100_sample.csv")

# Jittered in BOTH x and y (the original only jittered x) -- with n in the
# thousands and only 5 possible y-values, an x-only jitter stacks every
# point at a given (ClinVar tier, pipeline tier) pair into a single flat
# bar, hiding the individual points entirely instead of showing them.
rng = np.random.default_rng(0)
n = len(plot_df)
jitter_x = rng.uniform(-0.32, 0.32, size=n)
jitter_y = rng.uniform(-0.32, 0.32, size=n)

fig, ax = plt.subplots(figsize=(7, 7))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

ax.scatter(plot_df["clinvar_tier"] + jitter_x, plot_df["pipeline_tier"] + jitter_y,
           s=14, alpha=0.28, color=COLOR_POINTS, linewidths=0,
           label=f"Individual variants (n={n})", zorder=2)

means = plot_df.groupby("clinvar_tier")["pipeline_tier"].mean()
ax.plot(means.index, means.values, "o-", color=COLOR_MEAN, linewidth=2.5,
        markersize=9, markeredgecolor="white", markeredgewidth=1,
        label="Mean pipeline tier per ClinVar class", zorder=4)

ax.plot([1, 5], [1, 5], "--", color=COLOR_IDEAL, linewidth=1.3,
        label="Ideal (y = x)", zorder=1)

_labels = ["Benign", "Likely\nBenign", "VUS", "Likely\nPathogenic", "Pathogenic"]
ax.set_xticks([1, 2, 3, 4, 5]); ax.set_xticklabels(_labels)
ax.set_yticks([1, 2, 3, 4, 5]); ax.set_yticklabels(_labels)
ax.set_xlim(0.4, 5.6); ax.set_ylim(0.4, 5.6)
ax.set_xlabel("ClinVar classification", color="#52514e")
ax.set_ylabel("CardioClassifier classification", color="#52514e")
ax.set_title(f"Pipeline vs ClinVar concordance (n={n})", color="#0b0b0b", fontsize=13)

ax.grid(True, color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for _spine in ("top", "right"):
    ax.spines[_spine].set_visible(False)
for _spine in ("left", "bottom"):
    ax.spines[_spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")

ax.legend(loc="upper left", fontsize=9, frameon=False)
ax.set_aspect("equal")
fig.tight_layout()
fig.savefig("outputs/validation_concordance_plot.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

print(f"Exact match rate: {(plot_df['clinvar_tier'] == plot_df['pipeline_tier']).mean():.1%}")
print(f"Same broad category (P/LP vs B/LB) rate: "
      f"{((plot_df['clinvar_tier'] >= 4) == (plot_df['pipeline_tier'] >= 4)).mean():.1%}")


## Confusion matrix and diagnostic accuracy

The scatter-plot concordance view above is hard to read at a glance. The same comparison, shown as a plain 5x5 count table plus standard sensitivity/specificity/PPV/NPV (treating Pathogenic/Likely Pathogenic as "test positive" and Benign/Likely Benign as "test negative", ClinVar as ground truth), is the conventional way this exact comparison is presented in the genomic-medicine literature -- e.g. Whiffin et al. 2018 (the original CardioClassifier paper) report concordance purely as counts/percentages and a grouped bar chart, not a scatter plot.

If specificity is high (few false positives) but sensitivity is imperfect, that's a specific, well-documented pattern: clinicians and laboratories systematically disagree in the *same direction* -- Bland et al. (*Genet Med* 2018;20(3), PMID 29240077) found 18% of 688 laboratory-vs-clinician classifications discordant, and clinicians were the *more conservative* party in 66% of those, most often laboratory Pathogenic/Likely Pathogenic calls being read back as VUS. The most direct explanation for the same pattern here is the one already documented in this project's Limitations section: 7 of the 28 ACMG/AMP codes require patient- or family-specific evidence (segregation, de novo status, phenotype specificity) that a retrospective ClinVar pull cannot supply, so some ClinVar-Pathogenic calls that real submitters reached *with* that evidence are, correctly, less certain here without it.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

_TIER_LABELS = ["Benign", "Likely Benign", "VUS", "Likely Pathogenic", "Pathogenic"]
_CLINVAR_LABEL_MAP = {
    "benign": "Benign", "likely benign": "Likely Benign",
    "uncertain significance": "VUS", "vus": "VUS",
    "likely pathogenic": "Likely Pathogenic", "pathogenic": "Pathogenic",
}

cm_df = validation_results_df.copy()
cm_df["clinvar_tier_label"] = cm_df["clinvar_significance"].str.strip().str.lower().map(_CLINVAR_LABEL_MAP)
cm_df = cm_df.dropna(subset=["clinvar_tier_label"])

conf = pd.crosstab(cm_df["clinvar_tier_label"], cm_df["pipeline_classification"])
conf = conf.reindex(index=_TIER_LABELS, columns=_TIER_LABELS, fill_value=0)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

# log1p scale for the fill colour only (counts span >2 orders of magnitude
# across cells) -- the annotated numbers are always the real counts.
log_conf = np.log1p(conf.values)
ax.imshow(log_conf, cmap="Blues", vmin=0, extent=(-0.5, 4.5, 4.5, -0.5), aspect="equal")
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(4.5, -0.5)

for i in range(5):
    for j in range(5):
        count = conf.values[i, j]
        if count == 0:
            continue
        rel = log_conf[i, j] / (log_conf.max() or 1)
        ax.text(j, i, str(count), ha="center", va="center", fontsize=11,
                 color="#ffffff" if rel > 0.6 else "#0b0b0b")

ax.set_xticks(range(5)); ax.set_xticklabels(_TIER_LABELS, rotation=30, ha="right")
ax.set_yticks(range(5)); ax.set_yticklabels(_TIER_LABELS)
ax.set_xlabel("Pipeline classification", color="#52514e")
ax.set_ylabel("ClinVar classification", color="#52514e")
ax.set_title(f"Pipeline vs ClinVar: confusion matrix (n={len(cm_df)})", color="#0b0b0b", fontsize=13)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_xticks(np.arange(-0.5, 5, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 5, 1), minor=True)
ax.grid(which="minor", color="#fcfcfb", linewidth=2.5)
ax.tick_params(which="minor", length=0)
ax.tick_params(which="major", length=0)
fig.tight_layout()
fig.savefig("outputs/validation_confusion_matrix.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

# --------------------------------------------------
# Sensitivity / Specificity / PPV / NPV
# --------------------------------------------------
# Positive = ClinVar Pathogenic or Likely Pathogenic; Negative = ClinVar
# Benign or Likely Benign. A pipeline call of VUS counts as neither a hit
# nor an opposite-direction miss -- it's an abstention, not a wrong answer
# -- so it's excluded from the FN/FP counts and reported separately as an
# "inconclusive rate", matching how Josephs et al. 2023 frame retention of
# known P/LP variants (their "Set 1" sensitivity analysis) rather than
# forcing a strict binary outcome on every variant.
positive_clinvar = cm_df["clinvar_tier_label"].isin(["Pathogenic", "Likely Pathogenic"])
negative_clinvar = cm_df["clinvar_tier_label"].isin(["Benign", "Likely Benign"])
positive_pipeline = cm_df["pipeline_classification"].isin(["Pathogenic", "Likely Pathogenic"])
negative_pipeline = cm_df["pipeline_classification"].isin(["Benign", "Likely Benign"])
vus_pipeline = cm_df["pipeline_classification"] == "VUS"

TP = int((positive_clinvar & positive_pipeline).sum())
FN = int((positive_clinvar & (negative_pipeline | vus_pipeline)).sum())
TN = int((negative_clinvar & negative_pipeline).sum())
FP = int((negative_clinvar & positive_pipeline).sum())

sensitivity = TP / (TP + FN) if (TP + FN) else float("nan")
specificity = TN / (TN + FP) if (TN + FP) else float("nan")
ppv = TP / (TP + FP) if (TP + FP) else float("nan")
npv = TN / (TN + FN) if (TN + FN) else float("nan")
vus_rate_pos = (positive_clinvar & vus_pipeline).sum() / positive_clinvar.sum() if positive_clinvar.sum() else float("nan")
vus_rate_neg = (negative_clinvar & vus_pipeline).sum() / negative_clinvar.sum() if negative_clinvar.sum() else float("nan")

print(f"TP={TP}  FN={FN}  TN={TN}  FP={FP}")
print(f"Sensitivity: {sensitivity:.1%}")
print(f"Specificity: {specificity:.1%}")
print(f"PPV:         {ppv:.1%}")
print(f"NPV:         {npv:.1%}")
print(f"VUS (inconclusive) rate among ClinVar-Pathogenic/Likely Pathogenic: {vus_rate_pos:.1%}")
print(f"VUS (inconclusive) rate among ClinVar-Benign/Likely Benign:        {vus_rate_neg:.1%}")


## Zooming into the biggest miss: ClinVar-Pathogenic variants the pipeline calls VUS

Before building anything to *add* case-level evidence automatically, it's worth checking whether NCBI's ClinVar data actually contains usable case-level facts at all. It doesn't: pulling the full ClinVar record (`efetch`, `rettype=vcv`) for real variants -- including one curated by the **ClinGen Lysosomal Storage Disorder Expert Panel**, about as rigorous as ClinVar submissions get -- shows the structured case-level fields (`Origin`, `AffectedStatus`, observed-data `Description`) are consistently generic or blank ("germline"/"unknown"/"not provided"), even for expert-panel curations. There's no reliable field here to extract segregation, de novo status, or phasing from at scale; the only place any of that ever appears is unstructured prose in a `Comment` field, and even that's usually generic reasoning rather than concrete case facts. This confirms rather than closes the gap already described in the Limitations section -- the only real path to case-level evidence remains manual literature review of a small hand-picked subset.

Given that, the more useful thing to show here is *what's actually driving* the largest discordant cell in the confusion matrix above (ClinVar Pathogenic -> pipeline VUS) -- i.e. concretely, not abstractly, why these particular variants don't clear the posterior threshold.


In [ ]:
# --------------------------------------------------
# Breakdown of one confusion-matrix cell, by evidence code
# --------------------------------------------------
# Configurable so the same cell can inspect any discordant pair -- defaults
# to the biggest miss bucket (Pathogenic -> VUS).
from collections import Counter

# Defined here too (not just in the rule-activation cell further down) so
# this cell doesn't depend on running after it.
CASE_LEVEL_CODES = {"PM3", "PP1", "PP4", "BS2", "BS4", "BP2", "BP5", "PS2", "PM6"}

BREAKDOWN_CLINVAR_TIER = "Pathogenic"
BREAKDOWN_PIPELINE_TIER = "VUS"

miss_df = cm_df[
    (cm_df["clinvar_tier_label"] == BREAKDOWN_CLINVAR_TIER)
    & (cm_df["pipeline_classification"] == BREAKDOWN_PIPELINE_TIER)
].copy()


def _normalize_codes(codes):
    return tuple(sorted(c[0] if isinstance(c, list) else c for c in (codes or [])))


miss_df["code_combo"] = miss_df["evidence_codes"].map(_normalize_codes)
n_miss = len(miss_df)

print(f"{n_miss} variants: ClinVar {BREAKDOWN_CLINVAR_TIER} -> pipeline {BREAKDOWN_PIPELINE_TIER}")
print()
print("Most common exact evidence-code combinations:")
for combo, count in Counter(miss_df["code_combo"]).most_common(10):
    label = ", ".join(combo) if combo else "(none)"
    print(f"  {count:4d}  {label}")

miss_code_freq = Counter()
for combo in miss_df["code_combo"]:
    miss_code_freq.update(combo)

fig, ax = plt.subplots(figsize=(7, 4))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

miss_ordered = sorted(miss_code_freq, key=lambda c: -miss_code_freq[c])
miss_values = [miss_code_freq[c] / n_miss * 100 for c in miss_ordered]
miss_colors = ["#e87ba4" if c in CASE_LEVEL_CODES else "#6da7ec" for c in miss_ordered]

bars = ax.barh(range(len(miss_ordered)), miss_values, color=miss_colors, height=0.6)
xmax = max(miss_values) if miss_values else 1
for bar, v in zip(bars, miss_values):
    ax.text(bar.get_width() + xmax * 0.01, bar.get_y() + bar.get_height() / 2,
             f"{v:.0f}%", va="center", fontsize=9, color="#52514e")

ax.set_yticks(range(len(miss_ordered)))
ax.set_yticklabels(miss_ordered)
ax.invert_yaxis()
ax.set_xlim(0, xmax * 1.15)
ax.set_xlabel(f"% of the {n_miss} missed variants where code fired", color="#52514e")
ax.set_title(f"Evidence within ClinVar {BREAKDOWN_CLINVAR_TIER} → pipeline {BREAKDOWN_PIPELINE_TIER} (n={n_miss})",
             fontsize=11, color="#0b0b0b")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("left", "bottom"):
    ax.spines[spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
fig.tight_layout()
fig.savefig("outputs/validation_miss_breakdown.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

no_case_level = sum(1 for combo in miss_df["code_combo"] if not (set(combo) & CASE_LEVEL_CODES))
print(f"\n{no_case_level}/{n_miss} ({no_case_level/n_miss:.0%}) of these have ZERO case-level evidence "
      f"(expected -- confirmed above that none fire in this dataset at all).")


### How close were these misses to Likely Pathogenic?

Same idea applied to the 126 Pathogenic -> VUS misses above: bucket them by Tavtigian points into VUS-low/mid/high (Tavtigian et al. 2020, Table 3) and see how many were genuinely far off vs. one piece of supporting-level evidence away from crossing the 6-point Likely Pathogenic line.


In [ ]:
# --------------------------------------------------
# VUS sub-band breakdown of the miss bucket (Tavtigian et al. 2020, Table 3)
# --------------------------------------------------
def _vus_band(score):
    if score <= 1:
        return "VUS-low (0-1 pts)"
    if score <= 3:
        return "VUS-mid (2-3 pts)"
    return "VUS-high (4-5 pts)"


# classify_variant computes vus_subtier itself now (classifierr.ipynb) --
# use it directly when present; fall back to a local recompute (same
# formula) for batches collected before that field existed.
if "vus_subtier" in miss_df.columns and miss_df["vus_subtier"].notna().any():
    miss_df["vus_band"] = miss_df["vus_subtier"] + miss_df["tavtigian_score"].map(
        lambda s: " (0-1 pts)" if s <= 1 else (" (2-3 pts)" if s <= 3 else " (4-5 pts)")
    )
else:
    miss_df["vus_band"] = miss_df["tavtigian_score"].map(_vus_band)
_band_order = ["VUS-low (0-1 pts)", "VUS-mid (2-3 pts)", "VUS-high (4-5 pts)"]
_band_counts = miss_df["vus_band"].value_counts().reindex(_band_order, fill_value=0)

print(f"VUS sub-band breakdown of the {n_miss} ClinVar-Pathogenic -> pipeline-VUS misses:")
for band, count in _band_counts.items():
    print(f"  {band}: {count:3d} ({count / n_miss:.0%})")

fig, ax = plt.subplots(figsize=(6, 3.5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")
_band_colors = ["#f3d9a4", "#e0a94a", "#a8660a"]
bars = ax.barh(range(len(_band_order)), _band_counts.values, color=_band_colors, height=0.55)
for bar, count in zip(bars, _band_counts.values):
    ax.text(bar.get_width() + max(_band_counts.values) * 0.015, bar.get_y() + bar.get_height() / 2,
             f"{count} ({count / n_miss:.0%})", va="center", fontsize=9, color="#52514e")
ax.set_yticks(range(len(_band_order)))
ax.set_yticklabels(_band_order)
ax.invert_yaxis()
ax.set_xlim(0, max(_band_counts.values) * 1.25)
ax.set_xlabel(f"Number of the {n_miss} missed variants", color="#52514e")
ax.set_title("How close were the Pathogenic misses to Likely Pathogenic (6 pts)?", fontsize=11, color="#0b0b0b")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("left", "bottom"):
    ax.spines[spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
fig.tight_layout()
fig.savefig("outputs/validation_miss_vus_bands.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

n_high = int(_band_counts["VUS-high (4-5 pts)"])
print(f"\n{n_high}/{n_miss} ({n_high/n_miss:.0%}) of the misses sit in VUS-high -- one Supporting-level "
      f"code (e.g. PP1 segregation, PP4 phenotype specificity) short of crossing into Likely Pathogenic. "
      f"These are the variants most likely to flip with real case-level evidence.")


## Posterior probability distribution

The discrete 5-tier confusion matrix above hides how *close* a call was -- a posterior of 0.89 (just under the 0.90 Likely Pathogenic threshold) and a posterior of 0.11 land in the same "VUS" bucket but mean very different things. This shows the pipeline's continuous Tavtigian posterior, split by ClinVar's ground-truth call, demonstrating the underlying Bayesian scoring is doing real discriminative work, not just tripping tier thresholds.

Tavtigian et al. 2020 (the point-scale companion paper to the 2018 Bayesian framework) score "Uncertain" as points 0-5 -- a six-point-wide band, versus 1-4 points for every other tier. The shading below splits VUS into **VUS-low (0-1 pts)**, **VUS-mid (2-3 pts)**, and **VUS-high (4-5 pts)**, using the boundaries the pipeline's own scoring formula would place there. This is not a distinction the paper names explicitly, but it falls directly out of Table 3's point ranges, and it's the natural way to ask "how close" a VUS call actually was -- exactly the question the shape of the histogram below raises on its own.


In [ ]:
hist_df = validation_results_df.copy()
hist_df["clinvar_group"] = hist_df["clinvar_significance"].str.strip().str.lower().map(
    lambda s: "Pathogenic" if "pathogenic" in s else ("Benign" if "benign" in s else None)
)
hist_df = hist_df.dropna(subset=["clinvar_group"])

fig, ax = plt.subplots(figsize=(7.5, 5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

# --------------------------------------------------
# VUS-low / VUS-mid / VUS-high shading
# --------------------------------------------------
# Tavtigian et al. 2020 (Table 3) scores "Uncertain" as points 0-5 -- six
# points wide, vs. 1-4 points for every other tier. Rather than pick an
# arbitrary posterior split, derive the two internal boundaries (2 and 4
# points) from the pipeline's own Bayesian formula (calculate_tavtigian_score
# in classifierr.ipynb: O_PP=2.08, prior=0.10), so the shading lines up with
# where a real variant's integer score actually places it.
_O_PP, _PRIOR = 2.08, 0.10


def _posterior_at(n_points):
    odds = _O_PP ** n_points
    return (odds * _PRIOR) / (odds * _PRIOR + (1 - _PRIOR))


_vus_lo, _vus_hi = _posterior_at(2), _posterior_at(4)  # 0.325, 0.675
_band_spans = [(0.10, _vus_lo, "VUS-low (0-1 pts)"),
               (_vus_lo, _vus_hi, "VUS-mid (2-3 pts)"),
               (_vus_hi, 0.90, "VUS-high (4-5 pts)")]
for i, (lo, hi, _) in enumerate(_band_spans):
    ax.axvspan(lo, hi, color="#c1830f", alpha=0.06 + i * 0.06, zorder=0)

bins = np.linspace(0, 1, 41)
_hist_colors = {"Benign": "#6da7ec", "Pathogenic": "#e87ba4"}
for grp, color in _hist_colors.items():
    vals = hist_df.loc[hist_df["clinvar_group"] == grp, "posterior_probability"]
    ax.hist(vals, bins=bins, alpha=0.55, color=color, label=f"ClinVar {grp} (n={len(vals)})", edgecolor="none")

for x in (0.001, 0.10, 0.90, 0.99):
    ax.axvline(x, color="#b4b3ac", linewidth=1, linestyle="--", zorder=0)
for x in (_vus_lo, _vus_hi):
    ax.axvline(x, color="#c1830f", linewidth=0.8, linestyle=":", zorder=0)

ax.set_yscale("log")
ax.set_ylim(bottom=0.7)
# Band names sit below the x-axis (axes-fraction y, data-coord x) so they
# never compete with the legend or the bars for space.
for lo, hi, label in _band_spans:
    ax.text((lo + hi) / 2, -0.065, label, ha="center", va="top", fontsize=8,
             color="#8a5a06", transform=ax.get_xaxis_transform())

ax.set_xlabel("Pipeline posterior probability of pathogenicity", color="#52514e", labelpad=22)
ax.set_ylabel("Number of variants (log scale)", color="#52514e")
ax.set_title(f"Posterior probability distribution by ClinVar category (n={len(hist_df)})",
             color="#0b0b0b", fontsize=12)
ax.grid(True, axis="y", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("left", "bottom"):
    ax.spines[spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
ax.legend(loc="upper center", fontsize=9, frameon=False)
fig.tight_layout()
fig.savefig("outputs/validation_posterior_histogram.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

print(f"VUS-low  [0.10 , {_vus_lo:.3f}) -- 0-1 Tavtigian points")
print(f"VUS-mid  [{_vus_lo:.3f}, {_vus_hi:.3f}) -- 2-3 Tavtigian points")
print(f"VUS-high [{_vus_hi:.3f}, 0.90 ) -- 4-5 Tavtigian points -- one Supporting-level code short of Likely Pathogenic (6 pts)")


## Which ACMG codes are actually firing?

Splits the 28 evidence codes into **case-level** (PM3, PP1, PP4, BS2, BS4, BP2, BP5, PS2, PM6 -- require patient/family information no retrospective ClinVar pull can supply) and **computational** (the other 19, retrievable from VEP/gnomAD/ClinGen CSpec/local functional-study curation alone), mirroring the same split Whiffin et al. 2018 draw in their own Figure 3a. This is the direct evidence behind the Limitations discussion: it's not that case-level codes are rare here, they are **measured and confirmed at 0%** for every one of them, which is the real reason some ClinVar-Pathogenic variants can't clear the posterior threshold without that missing evidence.


In [ ]:
from collections import Counter

CASE_LEVEL_CODES = {"PM3", "PP1", "PP4", "BS2", "BS4", "BP2", "BP5", "PS2", "PM6"}
ALL_ACMG_CODES = ["PVS1", "PS1", "PS2", "PS3", "PS4", "PM1", "PM2", "PM3", "PM4", "PM5", "PM6",
                   "PP1", "PP2", "PP3", "PP4", "PP5", "BA1", "BS1", "BS2", "BS3", "BS4",
                   "BP1", "BP2", "BP3", "BP4", "BP5", "BP6", "BP7"]


def rule_activation_frequencies(df):
    n = len(df)
    counts = Counter()
    for codes in df["evidence_codes"]:
        for c in (codes or []):
            code = c[0] if isinstance(c, list) else c
            counts[code] += 1
    return {code: counts.get(code, 0) / n for code in ALL_ACMG_CODES}


rule_freqs = rule_activation_frequencies(validation_results_df)
case_codes = sorted([c for c in ALL_ACMG_CODES if c in CASE_LEVEL_CODES], key=lambda c: -rule_freqs[c])
comp_codes = sorted([c for c in ALL_ACMG_CODES if c not in CASE_LEVEL_CODES], key=lambda c: -rule_freqs[c])
ordered_codes = case_codes + comp_codes
bar_colors = ["#e87ba4"] * len(case_codes) + ["#6da7ec"] * len(comp_codes)

fig, ax = plt.subplots(figsize=(7.5, 8))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

y_pos = np.arange(len(ordered_codes))
values = [rule_freqs[c] * 100 for c in ordered_codes]
bars = ax.barh(y_pos, values, color=bar_colors, height=0.65)
xmax = max(values) if max(values) > 0 else 1
for bar, v in zip(bars, values):
    ax.text(bar.get_width() + xmax * 0.01, bar.get_y() + bar.get_height() / 2,
             f"{v:.1f}%" if v > 0 else "0%", va="center", fontsize=8, color="#52514e")

ax.set_yticks(y_pos)
ax.set_yticklabels(ordered_codes)
ax.invert_yaxis()
ax.axhline(len(case_codes) - 0.5, color="#898781", linewidth=1)
ax.set_xlim(0, xmax * 1.12)

ax.set_xlabel("% of variants where code fired", color="#52514e")
ax.set_title(f"ACMG evidence code activation frequency (n={len(validation_results_df)})\n"
             "rose = case-level (patient-specific), blue = computational", fontsize=11, color="#0b0b0b")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("left", "bottom"):
    ax.spines[spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
fig.tight_layout()
fig.savefig("outputs/validation_rule_activation.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()


## Tier 1 figure: automated rules only

The chart above makes the case-level gap visible by showing both groups side by side (case-level all at 0%). For a standalone "automated baseline" figure -- the one to actually present as Tier 1 -- showing 9 empty rose bars is just noise; this drops them entirely and shows only the 19 computational codes that are doing real work in this batch.


In [ ]:
# --------------------------------------------------
# Tier 1: automated-rule activation frequency only
# --------------------------------------------------
# Same rule_activation_frequencies() computation as above, just restricted
# to the 19 computational codes and re-sorted by frequency alone (no
# case-level/computational split needed when case-level isn't shown at all).
automated_freqs = {c: f for c, f in rule_freqs.items() if c not in CASE_LEVEL_CODES}
automated_ordered = sorted(automated_freqs, key=lambda c: -automated_freqs[c])

fig, ax = plt.subplots(figsize=(7.5, 6))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

y_pos = np.arange(len(automated_ordered))
values = [automated_freqs[c] * 100 for c in automated_ordered]
bars = ax.barh(y_pos, values, color="#6da7ec", height=0.65)
xmax = max(values) if max(values) > 0 else 1
for bar, v in zip(bars, values):
    ax.text(bar.get_width() + xmax * 0.01, bar.get_y() + bar.get_height() / 2,
             f"{v:.1f}%" if v > 0 else "0%", va="center", fontsize=8, color="#52514e")

ax.set_yticks(y_pos)
ax.set_yticklabels(automated_ordered)
ax.invert_yaxis()
ax.set_xlim(0, xmax * 1.12)

ax.set_xlabel("% of variants where code fired", color="#52514e")
ax.set_title(f"Tier 1: automated ACMG rule activation frequency (n={len(validation_results_df)})",
             fontsize=12, color="#0b0b0b")
ax.grid(True, axis="x", color="#e1e0d9", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("left", "bottom"):
    ax.spines[spine].set_color("#c3c2b7")
ax.tick_params(colors="#898781")
fig.tight_layout()
fig.savefig("outputs/validation_rule_activation_automated_only.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()


## Manual-curation subset: does real case-level evidence resolve the undercalls?

Mirrors Whiffin et al. 2018's own validation design directly: rather than trying to source case-level evidence automatically (confirmed above not to exist in ClinVar's structured fields), they manually curated a small set of variants and reported the classification *before* vs *after* adding that evidence (their Figure 3a: 87.3% automated-only concordance -> 87.7% exact match after manual case-level curation).

The subset below is drawn from the 126 Pathogenic -> VUS undercalls, stratified across genes (up to 3 per gene) rather than all from one gene, so the result generalises across the panel rather than being a single-gene anecdote. **Note:** 3 of the FLNC variants show `(FLNC-AS1)` in their HGVS string -- this is just how ClinVar's `genes` field happened to be recorded for these specific records (FLNC-AS1 is an antisense-RNA gene overlapping FLNC); the transcript accession (NM_001458.5) is FLNC's own, and the pipeline itself already resolves these correctly to FLNC (confirmed via `context['gene_symbol']` in the original run). Search ClinVar/PubMed for these using the gene **FLNC**, not FLNC-AS1.


In [ ]:
# --------------------------------------------------
# Select the subset and write a fill-in template
# --------------------------------------------------
import random

CURATION_TARGET = 18
CURATION_PER_GENE_CAP = 3
CURATION_SEED = 0

miss_all = cm_df[
    (cm_df["clinvar_tier_label"] == "Pathogenic") & (cm_df["pipeline_classification"] == "VUS")
].copy()

by_gene = {gene: g.to_dict("records") for gene, g in miss_all.groupby("gene")}
genes_by_volume = sorted(by_gene, key=lambda g: -len(by_gene[g]))

_rng = random.Random(CURATION_SEED)
curation_selection = []
for gene in genes_by_volume:
    if len(curation_selection) >= CURATION_TARGET:
        break
    take = min(CURATION_PER_GENE_CAP, len(by_gene[gene]), CURATION_TARGET - len(curation_selection))
    curation_selection.extend(_rng.sample(by_gene[gene], take))

print(f"{len(curation_selection)} variants selected across "
      f"{len(set(r['gene'] for r in curation_selection))} genes")

template_rows = []
for r in curation_selection:
    template_rows.append({
        "hgvs": r["hgvs"],
        "gene": r["gene"],
        "clinvar_search_url": f"https://www.ncbi.nlm.nih.gov/clinvar/?term={r['hgvs']}",
        "current_pipeline_classification": r["pipeline_classification"],
        "current_posterior_probability": r["posterior_probability"],
        "current_evidence_codes": ", ".join(
            c[0] if isinstance(c, list) else c for c in r["evidence_codes"]
        ),
        # -------- fill these in from ClinVar's cited literature / submitter comments --------
        "segregation_answer": "",       # y / n / unknown -- was segregation assessed across family members?
        "segregation_result": "",       # segregates / does_not_segregate (only if segregation_answer=y)
        "segregation_meioses": "",      # number of informative meioses, if stated (blank if unknown)
        "de_novo_answer": "",           # y / n / unknown -- confirmed de novo?
        "parentage_answer": "",         # y / n -- was biological parentage (maternity+paternity) confirmed? (only if de_novo_answer=y)
        "phasing_answer": "",           # y / n / unknown -- second (likely) pathogenic variant in same gene?
        "phasing_relationship": "",     # cis / trans (only if phasing_answer=y)
        "pp4_answer": "",               # y / n / unknown -- phenotype highly specific for this gene's disease?
        "bs2_answer": "",               # y / n / unknown -- observed in a phenotype-negative adult past expected onset?
        "bp5_answer": "",               # y / n / unknown -- alternate molecular basis found for the phenotype?
        "notes": "",                    # free text -- source (PMID), what you found, anything ambiguous
    })

template_df = pd.DataFrame(template_rows)
template_path = "outputs/manual_curation_template.csv"
template_df.to_csv(template_path, index=False)
print(f"Saved: {template_path}")
template_df[["hgvs", "gene", "current_pipeline_classification", "current_posterior_probability"]]


### How to fill in `outputs/manual_curation_template.csv`

Open it in Excel/Numbers/any text editor -- work through it at your own pace, no need to touch the notebook again until you're done with all 18. For each variant, use the `clinvar_search_url` link (opens the variant's real ClinVar page, showing its submitters and cited PubMed IDs) to find real case-level facts:

- **Segregation is the most likely to be findable** -- ClinVar submitter comments and cited papers sometimes report "segregated with disease in N affected relatives." If you find a count, fill in both `segregation_answer=y`, `segregation_result=segregates` (or `does_not_segregate`), and `segregation_meioses` (the pipeline grades PP1 by this: Strong >=7, Moderate >=5, Supporting >=3 -- leave blank if the paper doesn't give a number, it'll still fire at a default "Supporting").
- **De novo status** is the next most likely -- look for "de novo," "confirmed de novo," or "maternity/paternity confirmed" in the citations.
- **Phasing (PM3/BP2), PP4, BS2, BP5** are much rarer to find from a quick literature check -- leave blank (= "unknown," which is the honest default, not a fabricated negative) if you don't find anything concrete. Don't guess.
- Leave a whole row entirely blank (all fill-in columns empty) if you can't find anything usable for that variant at all -- the batch cell below will just re-run it with no change, which is itself a valid, informative result (some variants may only be resolvable with real family/lab data that isn't published anywhere).

Any column left blank is read as "unknown" -- exactly the same honest default `run_variant_through_main1` already uses everywhere else in this notebook.


In [ ]:
# --------------------------------------------------
# Re-run the manually-curated subset and compare before/after
# --------------------------------------------------
# Reads back whatever you filled in to manual_curation_template.csv (blank
# cells -> "unknown", read the same honest way as everywhere else in this
# notebook) and re-runs each variant through the ACTUAL pipeline
# (run_variant_through_main1 -> %run -i mainn.ipynb) with that evidence
# supplied via extra_answers, exactly mirroring how a real interactive
# session would take it. This is NOT a separate scoring shortcut -- it's
# the same code path, just with real facts instead of "unknown" for these
# 18 variants specifically.
manual_curation_df = pd.read_csv("outputs/manual_curation_template.csv", keep_default_na=False)

_OVERRIDE_FIELDS = ["segregation_answer", "segregation_result", "de_novo_answer",
                    "parentage_answer", "phasing_answer", "phasing_relationship",
                    "pp4_answer", "bs2_answer", "bp5_answer"]

before_after_rows = []
for _, row in manual_curation_df.iterrows():
    hgvs = row["hgvs"]
    gene = row["gene"]
    disease_code, _, _ = infer_disease_short_code(gene)

    extra_answers = {}
    for field in _OVERRIDE_FIELDS:
        val = str(row.get(field, "")).strip().lower()
        if val:
            extra_answers[field] = val
    meioses_val = str(row.get("segregation_meioses", "")).strip()
    if meioses_val:
        try:
            extra_answers["segregation_meioses"] = int(float(meioses_val))
        except ValueError:
            print(f"  Warning: could not parse segregation_meioses={meioses_val!r} for {hgvs}, ignoring")

    after = run_variant_through_main1(hgvs, disease_code, **extra_answers)

    before_after_rows.append({
        "hgvs": hgvs,
        "gene": gene,
        "before_classification": row["current_pipeline_classification"],
        "before_posterior": float(row["current_posterior_probability"]),
        "after_classification": after["pipeline_classification"],
        "after_posterior": after["posterior_probability"],
        "after_evidence_codes": ", ".join(
            c[0] if isinstance(c, list) else c for c in after["evidence_codes"]
        ),
        "evidence_added": ", ".join(f"{k}={v}" for k, v in extra_answers.items()) or "(none found)",
        "reclassified": row["current_pipeline_classification"] != after["pipeline_classification"],
    })

before_after_df = pd.DataFrame(before_after_rows)
before_after_df.to_csv("outputs/manual_curation_results.csv", index=False)

n_reclassified = int(before_after_df["reclassified"].sum())
n_with_evidence = int((before_after_df["evidence_added"] != "(none found)").sum())
print(f"{n_with_evidence}/{len(before_after_df)} variants had usable evidence entered")
print(f"{n_reclassified}/{len(before_after_df)} variants reclassified after adding it")
print("Saved: outputs/manual_curation_results.csv")

before_after_df[["hgvs", "gene", "before_classification", "after_classification",
                  "before_posterior", "after_posterior", "evidence_added"]]
